# Fragmentación del árbol coronario durante el preprocesado

**Nota metodológica sobre la primera versión de este notebook.** En ella se comparaba `nnUNet_raw/labelsTr` con `nnUNet_preprocessed/.../gt_segmentations`. Sin embargo, `gt_segmentations` no es la máscara remuestreada al spacing de entrenamiento, sino una copia en resolución original que nnU-Net conserva aparte para calcular las métricas finales, por lo que los valores resultaban idénticos: se comparaba el dato crudo consigo mismo.

**Esta versión carga la segmentación que realmente recibe el modelo durante el entrenamiento**, almacenada en formato blosc2 en `nnUNetPlans_3d_fullres/{case_id}_seg.b2nd` (ubicación confirmada mediante inspección directa de la carpeta en Drive).

Sobre los mismos 40 casos de validación se comparan:
- Número de componentes conexas: dato crudo frente a dato remuestreado.
- Cobertura de la componente principal respecto a su propio total (crudo frente a remuestreado).
- Volumen físico del vaso en mm³ (crudo frente a remuestreado). Si el remuestreo eliminara vaso fino, el volumen disminuiría además de fragmentarse.


## 1. Entorno


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q SimpleITK scikit-image scipy pandas blosc2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 30.5 MB/s eta 0:00:00


In [3]:
import os
from pathlib import Path

BASE = Path('/content/drive/MyDrive/TFM_nnUNet')
NNUNET_RAW = BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = BASE / 'nnUNet_preprocessed'

DATASET_FOLDER_NAME = 'Dataset501_ImageCAS'

GT_CRUDO = NNUNET_RAW / DATASET_FOLDER_NAME / 'labelsTr'
CARPETA_PREPROCESADA = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / 'nnUNetPlans_3d_fullres'

print('GT crudo:              ', GT_CRUDO, '-> existe:', GT_CRUDO.exists())
print('Carpeta preprocesada:  ', CARPETA_PREPROCESADA, '-> existe:', CARPETA_PREPROCESADA.exists())

if CARPETA_PREPROCESADA.exists():
    ejemplos = sorted(CARPETA_PREPROCESADA.glob('*_seg.b2nd'))[:3]
    print('Ejemplos de segmentación preprocesada:', [p.name for p in ejemplos])

GT crudo:               /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw/Dataset501_ImageCAS/labelsTr -> existe: True
Carpeta preprocesada:   /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset501_ImageCAS/nnUNetPlans_3d_fullres -> existe: True
Ejemplos de segmentación preprocesada: ['ImageCAS_0001_seg.b2nd', 'ImageCAS_0002_seg.b2nd', 'ImageCAS_0003_seg.b2nd']


## 2. Casos a comparar (los mismos 40 de validación, fold 0)


In [4]:
import json as _json

SPLITS_PATH = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / 'splits_final.json'
splits = _json.loads(SPLITS_PATH.read_text())
casos_validacion = splits[0]['val']

print(f'{len(casos_validacion)} casos de validación (fold 0)')
print(casos_validacion[:5], '...')

40 casos de validación (fold 0)
['ImageCAS_0011', 'ImageCAS_0017', 'ImageCAS_0018', 'ImageCAS_0021', 'ImageCAS_0028'] ...


## 3. Funciones de carga y de métricas


In [5]:
import pickle
import numpy as np
import SimpleITK as sitk
import blosc2
from scipy import ndimage

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)


def cargar_gt_crudo(case_id):
    """Ground truth en nnUNet_raw, resolución/spacing original."""
    path = GT_CRUDO / f'{case_id}.nii.gz'
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]  # (x,y,z) -> (z,y,x)
    mask = (array > 0).astype(np.uint8)
    return mask, spacing


def cargar_gt_preprocesado(case_id):
    """Segmentación real remuestreada a la spacing de entrenamiento
    (la que ve el modelo), leída del .b2nd + spacing del .pkl asociado."""
    seg_path = CARPETA_PREPROCESADA / f'{case_id}_seg.b2nd'
    pkl_path = CARPETA_PREPROCESADA / f'{case_id}.pkl'

    seg_array = np.asarray(blosc2.open(str(seg_path))[:])
    if seg_array.ndim == 4:
        seg_array = seg_array[0]  # quitar canal si lo tiene

    with open(pkl_path, 'rb') as f:
        properties = pickle.load(f)

        # nnU-Net puede usar -1 como ignore_label: se conserva solo el foreground (> 0)
    mask = (seg_array > 0).astype(np.uint8)

        # El nombre de la clave de spacing puede variar según la versión;
        # se prueban las más habituales y se avisa si no se encuentra ninguna
    spacing = None
    for clave in ('spacing', 'spacing_after_resampling', 'current_spacing'):
        if clave in properties:
            spacing = properties[clave]
            break
    if spacing is None:
        print(f'[{case_id}] AVISO: no se encontró spacing en el .pkl. Claves disponibles: {list(properties.keys())}')
        spacing = (1.0, 1.0, 1.0)

    return mask, spacing, properties


def componentes_conexas(mask):
    labeled, n = ndimage.label(mask, structure=CONECTIVIDAD_26)
    return labeled, n


def cobertura_propia_componente_principal(mask):
    total = mask.sum()
    if total == 0:
        return 0.0, 0
    labeled, n = componentes_conexas(mask)
    tamanos = ndimage.sum(mask, labeled, index=range(1, n + 1))
    return tamanos.max() / total, n


def volumen_fisico_mm3(mask, spacing):
    voxel_volume = spacing[0] * spacing[1] * spacing[2]
    return mask.sum() * voxel_volume

## 4. Verificación

En primer lugar, se repite el test sintético anterior, que detecta una fragmentación deliberada. En segundo lugar, se carga un caso real por ambas vías y se muestran sus propiedades (formas y spacings), para confirmar que se están leyendo dos datos distintos y no el mismo dato dos veces, como ocurría en la primera versión.


In [6]:
zz, yy, xx = np.mgrid[0:40, 0:15, 0:15]
tubo = (((yy - 7) ** 2 + (xx - 7) ** 2) <= 4 ** 2).astype(np.uint8)

cob_intacto, n_intacto = cobertura_propia_componente_principal(tubo)
assert n_intacto == 1 and abs(cob_intacto - 1.0) < 1e-6

tubo_roto = tubo.copy()
tubo_roto[19:21, :, :] = 0
cob_roto, n_roto = cobertura_propia_componente_principal(tubo_roto)
assert n_roto == 2 and 0.4 < cob_roto < 0.6
print('OK: función de fragmentación verificada sobre datos sintéticos.')

OK: función de fragmentación verificada sobre datos sintéticos.


In [7]:
caso_prueba = casos_validacion[0]

mask_crudo, spacing_crudo = cargar_gt_crudo(caso_prueba)
mask_prep, spacing_prep, props = cargar_gt_preprocesado(caso_prueba)

print(f'--- {caso_prueba} ---')
print('CRUDO       -> shape:', mask_crudo.shape, ' spacing:', spacing_crudo, ' voxels FG:', mask_crudo.sum())
print('PREPROCESADO-> shape:', mask_prep.shape, ' spacing:', spacing_prep, ' voxels FG:', mask_prep.sum())
print()
print('Claves disponibles en el .pkl:', list(props.keys()))

assert mask_crudo.shape != mask_prep.shape or not np.allclose(spacing_crudo, spacing_prep), (
    'Las formas/spacings son idénticos — probablemente seguimos leyendo la misma '
    'resolución dos veces. Revisa las rutas antes de continuar con los 40 casos.'
)
print('\nOK: crudo y preprocesado tienen formas/spacings distintos, como se esperaba.')

--- ImageCAS_0011 ---
CRUDO       -> shape: (275, 512, 512)  spacing: (0.5, 0.330078125, 0.330078125)  voxels FG: 70200
PREPROCESADO-> shape: (275, 481, 481)  spacing: [np.float64(0.5), np.float64(0.330078125), np.float64(0.330078125)]  voxels FG: 62112

Claves disponibles en el .pkl: ['sitk_stuff', 'spacing', 'shape_before_cropping', 'bbox_used_for_cropping', 'shape_after_cropping_and_before_resampling', 'class_locations']

OK: crudo y preprocesado tienen formas/spacings distintos, como se esperaba.


## 5. Cálculo de métricas para los 40 casos


In [8]:
import pandas as pd

filas = []

for case_id in casos_validacion:
    fila = {'case_id': case_id}
    try:
        mask_crudo, spacing_crudo = cargar_gt_crudo(case_id)
        cob_crudo, n_crudo = cobertura_propia_componente_principal(mask_crudo)
        fila['n_componentes_crudo'] = n_crudo
        fila['cobertura_propia_crudo'] = cob_crudo
        fila['volumen_mm3_crudo'] = volumen_fisico_mm3(mask_crudo, spacing_crudo)
    except Exception as e:
        print(f'[{case_id}] ERROR cargando crudo: {e}')

    try:
        mask_prep, spacing_prep, _ = cargar_gt_preprocesado(case_id)
        cob_prep, n_prep = cobertura_propia_componente_principal(mask_prep)
        fila['n_componentes_preprocesado'] = n_prep
        fila['cobertura_propia_preprocesado'] = cob_prep
        fila['volumen_mm3_preprocesado'] = volumen_fisico_mm3(mask_prep, spacing_prep)
    except Exception as e:
        print(f'[{case_id}] ERROR cargando preprocesado: {e}')

    filas.append(fila)

df_diagnostico = pd.DataFrame(filas)
print(f'\n{len(df_diagnostico)} casos procesados')
df_diagnostico.head()


40 casos procesados


,case_id,n_componentes_crudo,cobertura_propia_crudo,volumen_mm3_crudo,n_componentes_preprocesado,cobertura_propia_preprocesado,volumen_mm3_preprocesado
0,ImageCAS_0011,3,0.539687,3824.200058,3,0.539429,3383.599915
1,ImageCAS_0017,2,0.524722,7559.638214,2,0.524806,11259.465790
2,ImageCAS_0018,2,0.651822,3924.680279,2,0.651390,3215.515495
3,ImageCAS_0021,2,0.683977,5250.521170,2,0.685377,5048.249393
4,ImageCAS_0028,2,0.506445,5480.980408,2,0.507577,5000.504486


## 6. Resultado del diagnóstico


In [9]:
columnas_resumen = [
    'n_componentes_crudo', 'n_componentes_preprocesado',
    'cobertura_propia_crudo', 'cobertura_propia_preprocesado',
    'volumen_mm3_crudo', 'volumen_mm3_preprocesado',
]
resumen_diagnostico = df_diagnostico[columnas_resumen].mean().round(4)
print('Medias sobre los', len(df_diagnostico), 'casos de validación:\n')
print(resumen_diagnostico)

print('\n--- Interpretación automática ---')
delta_componentes = resumen_diagnostico['n_componentes_preprocesado'] - resumen_diagnostico['n_componentes_crudo']
delta_cobertura = resumen_diagnostico['cobertura_propia_preprocesado'] - resumen_diagnostico['cobertura_propia_crudo']
delta_volumen_pct = 100 * (resumen_diagnostico['volumen_mm3_preprocesado'] - resumen_diagnostico['volumen_mm3_crudo']) / resumen_diagnostico['volumen_mm3_crudo']

print(f'Diferencia en nº de componentes (preprocesado - crudo): {delta_componentes:+.2f}')
print(f'Diferencia en cobertura propia (preprocesado - crudo): {delta_cobertura:+.4f}')
print(f'Cambio en volumen físico del vaso: {delta_volumen_pct:+.1f}%')

if delta_componentes > 2 or delta_cobertura < -0.05:
    print('\n>> El ground truth se fragmenta claramente AL REMUESTREAR.')
    print('>> El preprocesado es un candidato fuerte al problema de continuidad.')
elif abs(delta_componentes) <= 2 and abs(delta_cobertura) <= 0.05:
    print('\n>> El ground truth remuestreado se parece bastante al crudo.')
    print('>> El remuestreo no parece ser la causa principal de la fragmentación.')
else:
    print('\n>> Resultado mixto, revisar caso a caso antes de concluir.')

Medias sobre los 40 casos de validación:

n_componentes_crudo                 2.3000
n_componentes_preprocesado          2.3000
cobertura_propia_crudo              0.6101
cobertura_propia_preprocesado       0.6104
volumen_mm3_crudo                6664.3420
volumen_mm3_preprocesado         6982.2610
dtype: float64

--- Interpretación automática ---
Diferencia en nº de componentes (preprocesado - crudo): +0.00
Diferencia en cobertura propia (preprocesado - crudo): +0.0003
Cambio en volumen físico del vaso: +4.8%

>> El ground truth remuestreado se parece bastante al crudo.
>> El remuestreo no parece ser la causa principal de la fragmentación.


## 7. Test pareado de significación de la diferencia


In [10]:
from scipy import stats

pareja_componentes = df_diagnostico[['n_componentes_crudo', 'n_componentes_preprocesado']].dropna()
pareja_cobertura = df_diagnostico[['cobertura_propia_crudo', 'cobertura_propia_preprocesado']].dropna()

if len(pareja_componentes) >= 5:
    w_stat, w_p = stats.wilcoxon(pareja_componentes['n_componentes_crudo'], pareja_componentes['n_componentes_preprocesado'])
    t_stat, t_p = stats.ttest_rel(pareja_componentes['n_componentes_crudo'], pareja_componentes['n_componentes_preprocesado'])
    print(f'Nº componentes  — Wilcoxon p={w_p:.4f}, t-test p={t_p:.4f}')

if len(pareja_cobertura) >= 5:
    w_stat, w_p = stats.wilcoxon(pareja_cobertura['cobertura_propia_crudo'], pareja_cobertura['cobertura_propia_preprocesado'])
    t_stat, t_p = stats.ttest_rel(pareja_cobertura['cobertura_propia_crudo'], pareja_cobertura['cobertura_propia_preprocesado'])
    print(f'Cobertura propia — Wilcoxon p={w_p:.4f}, t-test p={t_p:.4f}')

Nº componentes  — Wilcoxon p=1.0000, t-test p=1.0000
Cobertura propia — Wilcoxon p=0.1969, t-test p=0.2659


## 8. Casos individuales, ordenados por mayor pérdida de cobertura tras el remuestreo


In [11]:
if 'cobertura_propia_crudo' in df_diagnostico.columns and 'cobertura_propia_preprocesado' in df_diagnostico.columns:
    df_diagnostico['delta_cobertura'] = (
        df_diagnostico['cobertura_propia_preprocesado'] - df_diagnostico['cobertura_propia_crudo']
    )
    peores_casos = df_diagnostico.sort_values('delta_cobertura').head(10)
    peores_casos[['case_id', 'n_componentes_crudo', 'n_componentes_preprocesado',
                  'cobertura_propia_crudo', 'cobertura_propia_preprocesado', 'delta_cobertura']]

## 9. Guardado de resultados


In [12]:
df_diagnostico.to_csv(BASE / 'diagnostico_fragmentacion_raw_vs_preprocesado_v2.csv', index=False)
print('Guardado en', BASE / 'diagnostico_fragmentacion_raw_vs_preprocesado_v2.csv')

Guardado en /content/drive/MyDrive/TFM_nnUNet/diagnostico_fragmentacion_raw_vs_preprocesado_v2.csv
